# EMR / Spark + Iceberg Demo (Local Playground)

This notebook connects to the **local Spark cluster** (EMR simulation) and demonstrates:
1. Reading Iceberg tables from the **Nessie** catalog
2. Iceberg **time travel** and **schema evolution**
3. **MERGE INTO** (UPSERT) operations
4. **Nessie branch operations** — Git-like data versioning
5. Structured Streaming simulation (micro-batch from MinIO)

### Prerequisites
- Playground stack running: `docker compose --env-file .env.playground up -d`
- Glue job already run (writes initial Iceberg tables): see `glue_iceberg_demo.ipynb`
- Connect this notebook to the Spark cluster at `spark://localhost:7077`

### If running inside the Glue JupyterLab (port 8888)
The Spark kernel is pre-configured. Just run the cells.

### If running in a local Jupyter (outside Docker)
Install: `pip install pyspark==3.5 pyarrow`  
Set the endpoint env vars before launching Jupyter.


In [ ]:
# Cell 1 — Imports and config
import os
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# When running inside Docker, use container hostnames.
# When running from host machine, switch to localhost variants.
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',    'http://minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY',  'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY',  'minioadmin')
NESSIE_URI       = os.getenv('NESSIE_URI',        'http://nessie:19120/api/v2')
WAREHOUSE        = os.getenv('ICEBERG_WAREHOUSE', 's3a://iceberg-warehouse/')
SPARK_MASTER     = os.getenv('SPARK_MASTER',      'spark://spark-master:7077')

RUN_TS = datetime.now(timezone.utc).isoformat()
print(f'Config ready. Run timestamp: {RUN_TS}')

In [ ]:
# Cell 2 — Build SparkSession pointing at local Spark cluster
spark = (
    SparkSession.builder
    .appName('datahive-emr-notebook')
    .master(SPARK_MASTER)
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.nessie',           'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .config('spark.sql.catalog.nessie.uri',          NESSIE_URI)
    .config('spark.sql.catalog.nessie.ref',          'main')
    .config('spark.sql.catalog.nessie.warehouse',    WAREHOUSE)
    .config('spark.sql.catalog.nessie.io-impl',  'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.nessie.s3.endpoint',          MINIO_ENDPOINT)
    .config('spark.sql.catalog.nessie.s3.path-style-access', 'true')
    .config('spark.sql.catalog.nessie.s3.access-key-id',     MINIO_ACCESS_KEY)
    .config('spark.sql.catalog.nessie.s3.secret-access-key', MINIO_SECRET_KEY)
    .config('spark.sql.defaultCatalog', 'nessie')
    .config('spark.hadoop.fs.s3a.impl',  'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint',           MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',         MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',         MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access',  'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.checksum.validation', 'false')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark connected. Version:', spark.version)
print('Master:', spark.sparkContext.master)

In [ ]:
# Cell 3 — List all Iceberg tables in the Nessie catalog
print('=== Nessie catalog: nessie.datahive ===')
spark.sql('SHOW TABLES IN nessie.datahive').show()

In [ ]:
# Cell 4 — Read and inspect the products Iceberg table
products = spark.table('nessie.datahive.products')
print(f'Products: {products.count()} rows')
products.printSchema()
products.show(truncate=False)

In [ ]:
# Cell 5 — Iceberg Time Travel
# View all snapshots written to the orders table
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM nessie.datahive.orders.snapshots
    ORDER BY committed_at
""")
snapshots.show(truncate=False)

# Read the orders table AS OF a specific snapshot
first_snapshot = snapshots.orderBy('committed_at').first()
if first_snapshot:
    sid = first_snapshot['snapshot_id']
    print(f'Time-travelling to snapshot_id={sid}')
    spark.sql(f"""
        SELECT COUNT(*) AS order_count_at_snapshot
        FROM nessie.datahive.orders
        VERSION AS OF {sid}
    """).show()

In [ ]:
# Cell 6 — MERGE INTO (UPSERT): update product stock quantities
stock_updates = spark.createDataFrame(
    [('P001', 220), ('P002', 175), ('P010', 50)],  # P010 is new
    schema=StructType([
        StructField('product_id',     StringType(),  False),
        StructField('stock_quantity', IntegerType(), True),
    ])
).withColumn('updated_at', F.lit(RUN_TS))

stock_updates.createOrReplaceTempView('stock_updates')

spark.sql("""
    MERGE INTO nessie.datahive.products AS t
    USING stock_updates AS s
    ON t.product_id = s.product_id
    WHEN MATCHED THEN
        UPDATE SET t.stock_quantity = s.stock_quantity,
                   t.updated_at = s.updated_at
    WHEN NOT MATCHED THEN
        INSERT (product_id, product_name, category, price, stock_quantity, updated_at, ingested_at)
        VALUES (s.product_id, 'New Product', 'Uncategorized', 0.0,
                s.stock_quantity, s.updated_at, s.updated_at)
""")

print('MERGE complete. Stock after update:')
spark.sql("""
    SELECT product_id, product_name, stock_quantity, updated_at
    FROM nessie.datahive.products
    WHERE product_id IN ('P001','P002','P010')
""").show(truncate=False)

In [ ]:
# Cell 7 — Nessie Branching (Git-like data versioning)

# Create an experimental branch
spark.sql('CREATE BRANCH IF NOT EXISTS dev_branch IN nessie')
print('Created branch: dev_branch')

# Switch to dev_branch and write a new table there
spark.sql('USE REFERENCE dev_branch IN nessie')

experimental = spark.createDataFrame(
    [('EXP001', 'Test Widget', 'Experimental', 5.99, 10)],
    schema=StructType([
        StructField('product_id',     StringType(),  False),
        StructField('product_name',   StringType(),  True),
        StructField('category',       StringType(),  True),
        StructField('price',          DoubleType(),  True),
        StructField('stock_quantity', IntegerType(), True),
    ])
)

experimental.writeTo('nessie.datahive.experimental_products').using('iceberg').createOrReplace()
print('Written experimental_products on dev_branch')

# Tables on dev_branch include experimental_products
print('Tables on dev_branch:')
spark.sql('SHOW TABLES IN nessie.datahive').show()

# Switch back to main — experimental_products is invisible there
spark.sql('USE REFERENCE main IN nessie')
print('Tables on main (experimental_products absent):')
spark.sql('SHOW TABLES IN nessie.datahive').show()

In [ ]:
# Cell 8 — Schema Evolution: add columns without rewriting data
spark.sql("""
    ALTER TABLE nessie.datahive.orders
    ADD COLUMNS (
        discount_code   STRING COMMENT 'Promotional code applied',
        is_gift         BOOLEAN COMMENT 'Whether order is a gift'
    )
""")
print('Added columns: discount_code, is_gift')

# Existing rows read NULL for new columns (no data rewrite)
spark.sql("""
    SELECT order_id, order_status, discount_code, is_gift
    FROM nessie.datahive.orders
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Cell 9 — Compaction (Iceberg maintenance)
# Merges small Parquet files into larger ones for better query performance
spark.sql('CALL nessie.system.rewrite_data_files(table => "datahive.orders")')
print('Compaction complete for nessie.datahive.orders')

# Expire old snapshots to save storage
from datetime import timedelta
cutoff = (datetime.now(timezone.utc) - timedelta(hours=1)).strftime('%Y-%m-%d %H:%M:%S')
spark.sql(f"""
    CALL nessie.system.expire_snapshots(
        table => 'datahive.orders',
        older_than => TIMESTAMP '{cutoff}',
        retain_last => 2
    )
""")
print(f'Expired snapshots older than {cutoff}')

In [ ]:
# Cell 10 — Spark Structured Streaming simulation
# Reads new JSON files dropped into raw-landing/events/ as a stream
# and aggregates event counts per event_type

from pyspark.sql.types import TimestampType

EVENT_SCHEMA = StructType([
    StructField('event_id',    StringType(),    True),
    StructField('event_type',  StringType(),    True),
    StructField('user_id',     StringType(),    True),
    StructField('session_id',  StringType(),    True),
    StructField('timestamp',   StringType(),    True),
])

stream_df = (
    spark.readStream
    .schema(EVENT_SCHEMA)
    .json('s3a://raw-landing/events/')
)

agg_df = (
    stream_df
    .withColumn('ts', F.to_timestamp('timestamp'))
    .groupBy(
        F.window('ts', '5 minutes'),
        'event_type'
    )
    .count()
    .withColumnRenamed('count', 'event_count')
)

# Write to console for playground (in production, write to Iceberg)
query = (
    agg_df.writeStream
    .outputMode('complete')
    .format('console')
    .option('truncate', False)
    .trigger(once=True)  # trigger=once processes existing files and exits
    .start()
)
query.awaitTermination(60)
print('Streaming query complete')

## Summary

| Feature | Demonstrated |
|---------|-------------|
| Read Iceberg table | ✓ Cell 4 |
| Time Travel | ✓ Cell 5 |
| MERGE INTO (UPSERT) | ✓ Cell 6 |
| Nessie branch isolation | ✓ Cell 7 |
| Schema evolution | ✓ Cell 8 |
| Table compaction | ✓ Cell 9 |
| Structured Streaming | ✓ Cell 10 |

### Useful Links
- Spark Master UI: http://localhost:8080
- Nessie UI + Catalog: http://localhost:19120
- MinIO Console: http://localhost:9001
- Job 4040 (active): http://localhost:4040
